In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")

# Parse timestamps to UTC datetimes as required
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)
df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

# Aggregate metric.csv by (cmdb_id, kpi_name) using the full series (global)
metric_agg = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(count='count',
         min='min',
         p50=lambda x: x.quantile(0.5),
         p95=lambda x: x.quantile(0.95),
         max='max')
    .reset_index()
)

# Top 50 by count desc
metric_agg_top = metric_agg.sort_values('count', ascending=False).head(50)

# Aggregate trace.csv: unique (cmdb_id, trace_name) pairs with count and mean(value)
trace_agg = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'])['value']
    .agg(count='count', mean='mean')
    .reset_index()
)
trace_agg_top = trace_agg.sort_values('count', ascending=False).head(50)

# Aggregate log.csv: unique (cmdb_id, log_name) pairs with count and mean(value)
log_agg = (
    df_log
    .groupby(['cmdb_id', 'log_name'])['value']
    .agg(count='count', mean='mean')
    .reset_index()
)
log_agg_top = log_agg.sort_values('count', ascending=False).head(50)

# Distinct cmdb_id values present across all three files (union)
distinct_cmdbs = sorted(
    set(df_metric['cmdb_id'].unique()).union(
        set(df_trace['cmdb_id'].unique()),
        set(df_log['cmdb_id'].unique())
    )
)

# Display compact results: only requested aggregate columns, limited to 50 rows
metric_agg_top, trace_agg_top, log_agg_top, distinct_cmdbs

```
Out[1]:
```
```python
# Compose a plain-English summary string using the previously computed aggregates
# Reuse existing variables: metric_agg_top, trace_agg_top, log_agg_top, distinct_cmdbs

# Helper to safely fetch a row and format selected stats
def fmt_metric_row(df, cmdb, kpi):
    sel = df[(df['cmdb_id']==cmdb) & (df['kpi_name']==kpi)]
    if sel.empty:
        return f"{cmdb}.{kpi}: (not found)"
    r = sel.iloc[0]
    return (f"{cmdb}.{kpi} (count={int(r['count'])}, min={r['min']:.6g}, "
            f"p50={r['p50']:.6g}, p95={r['p95']:.6g}, max={r['max']:.6g})")

def fmt_log_row(df, cmdb, logn):
    sel = df[(df['cmdb_id']==cmdb) & (df['log_name']==logn)]
    if sel.empty:
        return f"{cmdb}.{logn}: (not found)"
    r = sel.iloc[0]
    return f"{cmdb}.{logn} (count={int(r['count'])}, mean={r['mean']:.6g})"

examples = []
examples.append(fmt_metric_row(metric_agg_top, 'carts', 'cpu'))
examples.append(fmt_metric_row(metric_agg_top, 'carts', 'mem'))
examples.append(fmt_metric_row(metric_agg_top, 'queue-master', 'diskio'))

log_examples = []
log_examples.append(fmt_log_row(log_agg_top, 'front-end', 'log.row_count'))
log_examples.append(fmt_log_row(log_agg_top, 'front-end', 'log.error_count'))

trace_note = "trace.csv: no trace aggregates found (empty result)."
metric_note = ("metric.csv: aggregated by (cmdb_id, kpi_name) over the full series; "
               "returned top 50 pairs sorted by count. Many pairs have count=25.")
log_note = ("log.csv: aggregated by (cmdb_id, log_name); examples shown below. "
            "Several components show log.row_count and log.error_count; error means are 0 in shown entries.")

summary = (
    "Summary of telemetry aggregates:\n\n"
    f"- {metric_note}\n"
    f"  Examples: {examples[0]}; {examples[1]}; {examples[2]}.\n\n"
    f"- {trace_note}\n\n"
    f"- {log_note}\n"
    f"  Examples: {log_examples[0]}; {log_examples[1]}.\n\n"
    f"- Distinct cmdb_id values across all files (count={len(distinct_cmdbs)}): {distinct_cmdbs}\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id    kpi_name  count           min           p50           p95           max
0          carts         cpu     25  1.129431e+00  1.462095e+00  2.561016e+00  3.551945e+00
1          carts  latency-50     25  1.096748e-02  1.389156e-02  1.505546e-02  1.512021e-02
2          carts  latency-90     25  2.234630e-02  2.357831e-02  2.471742e-02  2.480233e-02
3          carts         mem     25  2.026950e+08  2.057694e+08  2.068285e+08  2.069194e+08
4          carts      socket     25  9.366667e+00  1.020000e+01  1.098000e+01  1.108333e+01
..           ...         ...    ...           ...           ...           ...           ...
61      shipping         cpu     25  1.711569e-01  4.505851e-01  8.321758e-01  1.084079e+00
46  queue-master         cpu     25  1.680835e-01  4.042192e-01  8.630886e-01  1.099428e+00
47  queue-master      diskio     25  1.827496e+04  4.523112e+04  6.002509e+04  1.021889e+05
48  queue-master         mem     25  2.514142e+08  2.526904e+08  2.541731e+08  2.541788e+08
49  queue-master      socket     25  3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00

[50 rows x 7 columns], Empty DataFrame
Columns: [cmdb_id, trace_name, count, mean]
Index: [],        cmdb_id         log_name  count     mean
7    front-end    log.row_count     25  1912.64
6    front-end  log.error_count     25     0.00
5    catalogue    log.row_count     25   155.40
4    catalogue  log.error_count     25     0.00
12     payment  log.error_count     25     0.00
..         ...              ...    ...      ...
17  session-db    log.row_count      5     5.00
10   orders-db  log.error_count      2     0.00
11   orders-db    log.row_count      2     3.50
22     user-db  log.error_count      1     0.00
23     user-db    log.row_count      1     1.00

[24 rows x 4 columns], ['carts', 'carts-db', 'catalogue', 'catalogue-db', 'front-end', 'orders', 'orders-db', 'payment', 'queue-master', 'rabbitmq', 'rabbitmq-exporter', 'session-db', 'shipping', 'user', 'user-db'])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd

# Reuse existing df_metric (assumed loaded and timestamp-parsed). If not present, load and parse.
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# 1) Compute global p95 per (cmdb_id, kpi_name) using the full series (global threshold)
p95_table = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(global_p95=lambda x: x.quantile(0.95), global_count='count')
    .reset_index()
)

# 2) Define incident window (UTC) and filter metric rows inside the window
window_start = pd.to_datetime('2024-01-21 08:01:18', utc=True)
window_end = pd.to_datetime('2024-01-21 08:31:18', utc=True)

df_window = df_metric[(df_metric['timestamp'] >= window_start) & (df_metric['timestamp'] <= window_end)].copy()

# 3) Attach global p95 to window rows and find rows where value >= global_p95
df_window = df_window.merge(p95_table, on=['cmdb_id', 'kpi_name'], how='left')
df_anom_rows = df_window[df_window['value'] >= df_window['global_p95']].copy()

# 4) Aggregate anomalies per (cmdb_id, kpi_name)
if not df_anom_rows.empty:
    anomalies = (
        df_anom_rows
        .groupby(['cmdb_id', 'kpi_name'])
        .agg(
            anomaly_count=('value', 'count'),
            earliest_anomaly_timestamp=('timestamp', 'min'),
            max_value_in_window=('value', 'max'),
            global_p95=('global_p95', 'first')
        )
        .reset_index()
    )
    # convert earliest_anomaly_timestamp to Unix seconds (int)
    anomalies['earliest_anomaly_timestamp'] = (anomalies['earliest_anomaly_timestamp'].astype('int64') // 10**9)
    anomalies_top = anomalies.sort_values(['anomaly_count', 'max_value_in_window'], ascending=[False, False]).head(20)
else:
    # empty anomalies DataFrame with requested columns
    anomalies_top = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','global_p95'])

# 5) Provide the p95 table (first 50 rows) for inspection, sorted by global_count desc
p95_table_head50 = p95_table.sort_values('global_count', ascending=False).head(50)[['cmdb_id','kpi_name','global_p95','global_count']]

# Display compact results: anomalies (top 20) and p95 table head50
anomalies_top, p95_table_head50

```
Out[2]:
```
```python
# Summarize the anomaly detection results from the previous computation.
# Reuse existing variables: anomalies_top, p95_table_head50

# Prepare summary
if 'anomalies_top' in globals() and not anomalies_top.empty:
    n = anomalies_top.shape[0]
    top_n = anomalies_top.head(10)
    rows = []
    for _, r in top_n.iterrows():
        ts = pd.to_datetime(int(r['earliest_anomaly_timestamp']), unit='s', utc=True)
        rows.append(f"{r['cmdb_id']}.{r['kpi_name']} – anomalies={int(r['anomaly_count'])}, "
                    f"earliest={ts.strftime('%Y-%m-%d %H:%M:%S %Z')}, "
                    f"max={r['max_value_in_window']:.6g}, p95={r['global_p95']:.6g}")
    summary = (
        f"Anomalies found: {n} (showing up to 10 top rows)\n" +
        "\n".join(rows) +
        "\n\nNote: global p95 thresholds were computed from the full metric series. "
        "See p95 table (first 50 rows) for thresholds per (cmdb_id, kpi_name)."
    )
else:
    # No anomalies: still provide p95 table head for inspection
    p95_preview = []
    if 'p95_table_head50' in globals() and not p95_table_head50.empty:
        for _, r in p95_table_head50.head(10).iterrows():
            p95_preview.append(f"{r['cmdb_id']}.{r['kpi_name']} -> p95={r['global_p95']:.6g} (count={int(r['global_count'])})")
        p95_text = "\n".join(p95_preview)
    else:
        p95_text = "(p95 table is empty)"
    summary = (
        "No anomalies found in the incident window for any (cmdb_id, kpi_name).\n\n"
        "Global p95 thresholds (first 10 of 50 shown):\n" + p95_text
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id kpi_name  anomaly_count  earliest_anomaly_timestamp  max_value_in_window    global_p95
55  rabbitmq-exporter      mem             25                  1705824240         4.767744e+06  4.767744e+06
53           rabbitmq   socket             25                  1705824240         1.100000e+01  1.100000e+01
9            carts-db   socket             25                  1705824240         6.000000e+00  6.000000e+00
14          catalogue   socket             25                  1705824240         6.000000e+00  6.000000e+00
19       catalogue-db   socket             25                  1705824240         4.000000e+00  4.000000e+00
49       queue-master   socket             25                  1705824240         3.000000e+00  3.000000e+00
60         session-db   socket             25                  1705824240         3.000000e+00  3.000000e+00
56  rabbitmq-exporter   socket             25                  1705824240         1.000000e+00  1.000000e+00
17       catalogue-db   diskio             25                  1705824240         0.000000e+00  0.000000e+00
40            payment   diskio             25                  1705824240         0.000000e+00  0.000000e+00
51           rabbitmq   diskio             25                  1705824240         0.000000e+00  0.000000e+00
71               user   socket             23                  1705824360         2.100000e+01  2.100000e+01
76            user-db   socket             23                  1705824360         1.000000e+01  1.000000e+01
38          orders-db   socket             23                  1705824360         9.000000e+00  9.000000e+00
44            payment   socket             12                  1705825020         1.000000e+01  1.000000e+01
32             orders      mem              2                  1705824360         6.510923e+08  6.493091e+08
43            payment      mem              2                  1705825200         5.829857e+08  5.211366e+08
18       catalogue-db      mem              2                  1705824720         4.306600e+08  4.306451e+08
64           shipping      mem              2                  1705825560         3.057467e+08  3.057462e+08
48       queue-master      mem              2                  1705825260         2.541788e+08  2.541731e+08,          cmdb_id    kpi_name    global_p95  global_count
0          carts         cpu  2.561016e+00            25
1          carts  latency-50  1.505546e-02            25
2          carts  latency-90  2.471742e-02            25
3          carts         mem  2.068285e+08            25
4          carts      socket  1.098000e+01            25
..           ...         ...           ...           ...
61      shipping         cpu  8.321758e-01            25
46  queue-master         cpu  8.630886e-01            25
47  queue-master      diskio  6.002509e+04            25
48  queue-master         mem  2.541731e+08            25
49  queue-master      socket  3.000000e+00            25

[50 rows x 4 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd

# Reuse existing variables where available; otherwise load files
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Ensure anomalies_top exists (from previous computation). If not, create empty with expected columns.
if 'anomalies_top' not in globals():
    anomalies_top = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count',
                                         'earliest_anomaly_timestamp','max_value_in_window','global_p95'])

# Load or reuse log.csv
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")
    df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

# Load error_logs.csv
try:
    df_error_logs
except NameError:
    df_error_logs = pd.read_csv("error_logs.csv")
    df_error_logs['timestamp'] = pd.to_datetime(df_error_logs['timestamp'], unit='s', utc=True)

# Define selected cmdb_ids and kpi_names and incident window (UTC)
selected_cmdbs = ['carts','catalogue','orders','payment','user']
kpis_of_interest = ['socket','mem','cpu','diskio']
window_start = pd.to_datetime('2024-01-21 08:01:18', utc=True)
window_end = pd.to_datetime('2024-01-21 08:31:18', utc=True)

# 1) anomalies_top filtered to selected cmdb_ids (all columns)
anomalies_filtered = anomalies_top[anomalies_top['cmdb_id'].isin(selected_cmdbs)].copy()

# 2) metric.csv rows in window for selected cmdb_ids and kpi_name in kpis_of_interest
metric_window_filtered = (
    df_metric[
        (df_metric['timestamp'] >= window_start) &
        (df_metric['timestamp'] <= window_end) &
        (df_metric['cmdb_id'].isin(selected_cmdbs)) &
        (df_metric['kpi_name'].isin(kpis_of_interest))
    ]
    .loc[:, ['timestamp', 'cmdb_id', 'kpi_name', 'value']]
    .sort_values(['cmdb_id', 'timestamp'], ascending=[True, True])
    .head(200)
    .reset_index(drop=True)
)

# 3) log.csv rows in the same window for selected cmdb_ids
log_window_filtered = (
    df_log[
        (df_log['timestamp'] >= window_start) &
        (df_log['timestamp'] <= window_end) &
        (df_log['cmdb_id'].isin(selected_cmdbs))
    ]
    .loc[:, ['timestamp', 'cmdb_id', 'log_name', 'value']]
    .sort_values(['cmdb_id', 'timestamp'], ascending=[True, True])
    .head(200)
    .reset_index(drop=True)
)

# 4) error_logs.csv rows in the same window for selected cmdb_ids
error_logs_window_filtered = (
    df_error_logs[
        (df_error_logs['timestamp'] >= window_start) &
        (df_error_logs['timestamp'] <= window_end) &
        (df_error_logs['cmdb_id'].isin(selected_cmdbs))
    ]
    .loc[:, ['timestamp', 'cmdb_id', 'message']]
    .sort_values(['cmdb_id', 'timestamp'], ascending=[True, True])
    .head(200)
    .reset_index(drop=True)
)

# Return the four compact results
anomalies_filtered, metric_window_filtered, log_window_filtered, error_logs_window_filtered

```
Out[3]:
```
```python
# Compose a plain-English summary string from the previously computed results
# Reuse existing variables: anomalies_filtered, metric_window_filtered, log_window_filtered, error_logs_window_filtered

import pandas as pd

lines = []
lines.append("Summary for selected cmdb_ids = ['carts','catalogue','orders','payment','user'] (incident window 2024-01-21 08:01:18 to 08:31:18 UTC):")
lines.append("")

# 1) Anomalies summary
if 'anomalies_filtered' in globals() and not anomalies_filtered.empty:
    lines.append("Anomalies detected (per cmdb_id.kpi):")
    for _, r in anomalies_filtered.iterrows():
        ts = pd.to_datetime(int(r['earliest_anomaly_timestamp']), unit='s', utc=True)
        note = ""
        # highlight diskio p95 == 0 cases
        if float(r['global_p95']) == 0.0 and r['kpi_name'] == 'diskio':
            note = " (global p95=0 — threshold is zero)"
        lines.append(
            f"- {r['cmdb_id']}.{r['kpi_name']}: anomaly_count={int(r['anomaly_count'])}, "
            f"earliest={ts.strftime('%Y-%m-%d %H:%M:%S %Z')}, max_in_window={r['max_value_in_window']:.6g}, "
            f"global_p95={r['global_p95']:.6g}{note}"
        )
else:
    lines.append("No anomalies found for the selected cmdb_ids.")

lines.append("")

# 2) Metric rows returned
if 'metric_window_filtered' in globals():
    lines.append(f"Metric rows returned for selected cmdb_ids and kpis (socket, mem, cpu, diskio): {len(metric_window_filtered)} rows (limited to 200).")
    # show earliest and latest timestamps present in the returned slice
    if len(metric_window_filtered):
        first_ts = metric_window_filtered['timestamp'].min()
        last_ts = metric_window_filtered['timestamp'].max()
        lines.append(f"  Time range in returned metric rows: {first_ts} to {last_ts} (UTC).")
else:
    lines.append("Metric window slice not available.")

lines.append("")

# 3) Log rows returned
if 'log_window_filtered' in globals():
    lines.append(f"Log rows returned for selected cmdb_ids: {len(log_window_filtered)} rows (limited to 200).")
    # note whether any error_count > 0
    if not log_window_filtered.empty and 'log.error_count' in log_window_filtered['log_name'].unique():
        errs = log_window_filtered[(log_window_filtered['log_name']=='log.error_count') & (log_window_filtered['value']>0)]
        if errs.empty:
            lines.append("  Observed log.error_count entries are all zero in the returned slice.")
        else:
            lines.append(f"  Observed {len(errs)} log.error_count rows with value>0.")
else:
    lines.append("Log window slice not available.")

lines.append("")

# 4) error_logs
if 'error_logs_window_filtered' in globals():
    if error_logs_window_filtered.empty:
        lines.append("No error_log entries found for selected cmdb_ids in the incident window.")
    else:
        lines.append(f"Error log rows found: {len(error_logs_window_filtered)} (limited to 200).")
else:
    lines.append("error_logs data not available.")

lines.append("")
lines.append("Concise takeaways:")
lines.append("- catalogue.socket and user.socket show many socket anomalies (catalogue: 25, user: 23).")
lines.append("- payment.diskio shows 25 anomalies but global p95 for diskio is 0 (threshold is zero) — this likely reflects all-zero diskio series rather than a true spike.")
lines.append("- payment also has socket and mem anomalies (payment.socket 12, payment.mem 2).")
lines.append("- orders.mem shows 2 anomalies (small count) and several 'orders' metric rows in the window show high socket values.")
lines.append("- Logs slice contains rows but log.error_count appears zero in the returned slice; error_logs had no entries for these services in the window.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id kpi_name  anomaly_count  earliest_anomaly_timestamp  max_value_in_window    global_p95
14  catalogue   socket             25                  1705824240         6.000000e+00  6.000000e+00
40    payment   diskio             25                  1705824240         0.000000e+00  0.000000e+00
71       user   socket             23                  1705824360         2.100000e+01  2.100000e+01
44    payment   socket             12                  1705825020         1.000000e+01  1.000000e+01
32     orders      mem              2                  1705824360         6.510923e+08  6.493091e+08
43    payment      mem              2                  1705825200         5.829857e+08  5.211366e+08,                     timestamp cmdb_id kpi_name         value
0   2024-01-21 08:04:00+00:00   carts      cpu  1.150917e+00
1   2024-01-21 08:04:00+00:00   carts      mem  2.026950e+08
2   2024-01-21 08:04:00+00:00   carts   socket  9.738095e+00
3   2024-01-21 08:05:00+00:00   carts      cpu  1.574000e+00
4   2024-01-21 08:05:00+00:00   carts      mem  2.028044e+08
..                        ...     ...      ...           ...
195 2024-01-21 08:18:00+00:00  orders   socket  1.936667e+01
196 2024-01-21 08:19:00+00:00  orders      cpu  2.739431e+00
197 2024-01-21 08:19:00+00:00  orders      mem  3.311383e+08
198 2024-01-21 08:19:00+00:00  orders   socket  1.971667e+01
199 2024-01-21 08:20:00+00:00  orders      cpu  5.685802e+00

[200 rows x 4 columns],                     timestamp cmdb_id         log_name  value
0   2024-01-21 08:05:00+00:00   carts  log.error_count      0
1   2024-01-21 08:05:00+00:00   carts    log.row_count    125
2   2024-01-21 08:06:00+00:00   carts  log.error_count      0
3   2024-01-21 08:06:00+00:00   carts    log.row_count     75
4   2024-01-21 08:07:00+00:00   carts  log.error_count      0
..                        ...     ...              ...    ...
195 2024-01-21 08:04:00+00:00    user    log.row_count    536
196 2024-01-21 08:05:00+00:00    user  log.error_count      0
197 2024-01-21 08:05:00+00:00    user    log.row_count    644
198 2024-01-21 08:06:00+00:00    user  log.error_count      0
199 2024-01-21 08:06:00+00:00    user    log.row_count    577

[200 rows x 4 columns], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import numpy as np
import pandas as pd

# Reuse existing df_metric and p95_table if available; otherwise load/compute p95_table from full series (global)
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

try:
    p95_table
except NameError:
    # compute global p95 per (cmdb_id, kpi_name) from full series (allowed as it's global)
    p95_table = (
        df_metric
        .groupby(['cmdb_id', 'kpi_name'])['value']
        .agg(global_p95=lambda x: x.quantile(0.95), global_count='count')
        .reset_index()
    )

# Selected cmdb_ids and incident window (UTC)
selected_cmdbs = ['catalogue','user','orders','payment','carts']
window_start = pd.to_datetime('2024-01-21 08:01:18', utc=True)
window_end = pd.to_datetime('2024-01-21 08:31:18', utc=True)

# Filter metric to the incident window
df_window = df_metric[(df_metric['timestamp'] >= window_start) & (df_metric['timestamp'] <= window_end)].copy()

# Determine (cmdb_id, kpi_name) pairs to analyze: those present in metric.csv for the selected cmdbs
pairs = (
    p95_table[p95_table['cmdb_id'].isin(selected_cmdbs)]
    [['cmdb_id', 'kpi_name', 'global_p95']]
    .drop_duplicates()
    .reset_index(drop=True)
)

results = []
for _, row in pairs.iterrows():
    cmdb = row['cmdb_id']
    kpi = row['kpi_name']
    global_p95_val = float(row['global_p95'])

    # rows in window for this pair
    sub = df_window[(df_window['cmdb_id'] == cmdb) & (df_window['kpi_name'] == kpi)].copy()
    if sub.empty:
        # no data in window -> anomaly_count 0 (skip including later)
        anomaly_count = 0
        max_in_window = np.nan
        earliest_ts = np.nan
        longest_run = 0
    else:
        # determine anomaly mask: value >= global_p95 (use >= as requested)
        mask = sub['value'] >= global_p95_val
        anomaly_count = int(mask.sum())
        if anomaly_count == 0:
            max_in_window = float(sub['value'].max())
            earliest_ts = np.nan
            longest_run = 0
        else:
            # earliest anomaly timestamp (Unix seconds)
            earliest_ts = int(sub.loc[mask, 'timestamp'].min().to_datetime64().astype('int64') // 10**9)
            max_in_window = float(sub.loc[:, 'value'].max())

            # compute longest consecutive anomaly length
            # get anomaly timestamps as integer seconds sorted
            anom_ts = np.sort(sub.loc[mask, 'timestamp'].astype('int64') // 10**9)
            if anom_ts.size == 0:
                longest_run = 0
            else:
                diffs = np.diff(anom_ts)
                # consecutive minutes defined by diff == 60
                current = 1
                longest = 1
                for d in diffs:
                    if d == 60:
                        current += 1
                        if current > longest:
                            longest = current
                    else:
                        current = 1
                longest_run = int(longest)

    # breach ratio percent: 100*(max - p95)/p95, NaN if global_p95==0
    if global_p95_val == 0 or np.isnan(global_p95_val):
        breach_ratio = np.nan
    else:
        # if no data in window, we'll still compute using max_in_window (which may be NaN)
        breach_ratio = 100.0 * (max_in_window - global_p95_val) / global_p95_val

    results.append({
        'cmdb_id': cmdb,
        'kpi_name': kpi,
        'global_p95': global_p95_val,
        'anomaly_count': anomaly_count,
        'earliest_anomaly_timestamp': earliest_ts if not np.isnan(earliest_ts) else np.nan,
        'max_value_in_window': max_in_window,
        'breach_ratio_percent': breach_ratio,
        'longest_consecutive_anomaly_length': longest_run
    })

res_df = pd.DataFrame(results)

# Keep only rows with anomaly_count > 0
res_df = res_df[res_df['anomaly_count'] > 0].copy()

# Sort by breach_ratio_percent desc (NaN last), then anomaly_count desc
res_df['breach_sort'] = res_df['breach_ratio_percent'].fillna(-np.inf)  # ensure NaNs go to bottom
res_df = res_df.sort_values(['breach_sort', 'anomaly_count'], ascending=[False, False]).drop(columns=['breach_sort'])

# Limit to top 20 rows and present requested columns in compact form
final_cols = ['cmdb_id', 'kpi_name', 'global_p95', 'anomaly_count',
              'earliest_anomaly_timestamp', 'max_value_in_window',
              'breach_ratio_percent', 'longest_consecutive_anomaly_length']

result_top20 = res_df[final_cols].head(20).reset_index(drop=True)

result_top20

```
Out[4]:
```
Summary of results for cmdb_ids = ['catalogue','user','orders','payment','carts'] (window 2024-01-21 08:01:18–08:31:18 UTC), using global p95 thresholds:

- Total rows returned: 20 (only entries with anomaly_count > 0).
- Anomalies are generally brief (anomaly_count = 2 for most rows; longest consecutive runs are 1–2 minutes).

Top offenders by breach_ratio_percent:
1. orders.error — global_p95=2.4266, anomaly_count=2, earliest=1705824240, max=12.18571, breach_ratio ≈ 402%. (largest relative spike)
2. orders.workload — global_p95=4.1326, anomaly_count=2, earliest=1705824240, max=12.18571, breach_ratio ≈ 195%.
3. carts.cpu — global_p95=2.5610, anomaly_count=2, earliest=1705824360, max=3.551945, breach_ratio ≈ 38.7%.
4. orders.cpu — global_p95=41.60175, anomaly_count=2, earliest=1705824240, max=55.45416, breach_ratio ≈ 33.3%.
5. user.workload — global_p95=20.04745, anomaly_count=2, earliest=1705824240, max=24.91588, breach_ratio ≈ 24.3%.
6. user.cpu — global_p95=1.29248, anomaly_count=2, earliest=1705824240, max=1.531713, breach_ratio ≈ 18.5%.

Other notable entries:
- payment.mem and payment.cpu show modest breaches (breach_ratio ~11.9% and ~0.64% respectively), each with 2 anomaly rows.
- Several latency and workload KPIs across catalogue, user, payment, and orders show 2-minute brief breaches with smaller breach ratios (1–6% or a few percent).
- All anomaly_count values shown are small (mostly 2), and longest consecutive anomaly lengths are 1–2 minutes, indicating short spikes rather than prolonged degradation.

Interpretation:
- The largest relative anomalies are in orders (error and workload) — large spikes relative to their global p95 thresholds.
- Most anomalies are short-lived (1–2 minutes) and low in count; these look like brief spikes rather than sustained failure.
- Use the listed rows (global_p95, counts, earliest timestamps, max values, breach_ratio, and consecutive lengths) to guide further tracing/logging investigation into orders and carts first.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name    global_p95  anomaly_count  earliest_anomaly_timestamp  max_value_in_window  breach_ratio_percent  longest_consecutive_anomaly_length
0      orders       error  2.426600e+00              2                  1705824240         1.218571e+01            402.172352                                   2
1      orders    workload  4.132567e+00              2                  1705824240         1.218571e+01            194.870362                                   2
2       carts         cpu  2.561016e+00              2                  1705824360         3.551945e+00             38.692845                                   1
3      orders         cpu  4.160175e+01              2                  1705824240         5.545416e+01             33.297648                                   2
4        user    workload  2.004745e+01              2                  1705824240         2.491588e+01             24.284540                                   2
5        user         cpu  1.292479e+00              2                  1705824240         1.531713e+00             18.509665                                   2
6      orders  latency-90  3.148049e-01              2                  1705825140         3.691319e-01             17.257360                                   2
7     payment         mem  5.211366e+08              2                  1705825200         5.829857e+08             11.868105                                   1
8     payment  latency-50  3.193041e-03              2                  1705825380         3.402814e-03              6.569699                                   1
9      orders      socket  3.447667e+01              2                  1705824300         3.638333e+01              5.530310                                   2
10    payment  latency-90  2.201205e-01              2                  1705825140         2.306995e-01              4.806013                                   1
11     orders  latency-50  4.154541e-02              2                  1705825020         4.323540e-02              4.067833                                   1
12       user  latency-50  6.151822e-03              2                  1705824240         6.337694e-03              3.021420                                   2
13  catalogue  latency-90  4.844393e-03              2                  1705824240         4.930721e-03              1.782017                                   2
14       user  latency-90  9.297193e-03              2                  1705824240         9.411588e-03              1.230432                                   2
15      carts      socket  1.098000e+01              2                  1705824840         1.108333e+01              0.941105                                   1
16    payment    workload  2.126920e+00              2                  1705824420         2.144167e+00              0.810875                                   1
17  catalogue  latency-50  3.125088e-03              2                  1705824240         3.145708e-03              0.659803                                   2
18    payment         cpu  3.942738e+01              2                  1705825200         3.967992e+01              0.640514                                   1
19      carts  latency-50  1.505546e-02              2                  1705825620         1.512021e-02              0.430095                                   2```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse df_metric and p95_table if available; otherwise load/compute
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

try:
    p95_table
except NameError:
    p95_table = (
        df_metric
        .groupby(['cmdb_id', 'kpi_name'])['value']
        .agg(global_p95=lambda x: x.quantile(0.95))
        .reset_index()
    )

# Incident window (UTC)
window_start = pd.to_datetime('2024-01-21 08:01:18', utc=True)
window_end = pd.to_datetime('2024-01-21 08:31:18', utc=True)

# Specific KPI pairs to analyze
selected_pairs = [
    ('orders', 'error'),
    ('orders', 'workload'),
    ('orders', 'cpu'),
    ('user', 'socket'),
    ('catalogue', 'socket')
]
pairs_df = pd.DataFrame(selected_pairs, columns=['cmdb_id', 'kpi_name'])

# Filter metric to window first
df_window = df_metric[(df_metric['timestamp'] >= window_start) & (df_metric['timestamp'] <= window_end)].copy()

# 1) Time series rows for the selected KPIs: join with pairs_df to keep only those pairs
ts_rows = df_window.merge(pairs_df, on=['cmdb_id', 'kpi_name'], how='inner')
ts_rows_sorted = ts_rows.loc[:, ['timestamp', 'cmdb_id', 'kpi_name', 'value']].sort_values(['kpi_name', 'timestamp'], ascending=[True, True])
ts_rows_limited = ts_rows_sorted.head(200).reset_index(drop=True)

# 2) KPI-summary table: one row per KPI with global_p95, anomaly_count, earliest_anomaly_timestamp (Unix seconds or null), max_value_in_window
# Attach global_p95 from p95_table
pairs_with_p95 = pairs_df.merge(p95_table, on=['cmdb_id', 'kpi_name'], how='left')

summary_rows = []
for _, r in pairs_with_p95.iterrows():
    cmdb = r['cmdb_id']
    kpi = r['kpi_name']
    global_p95_val = float(r['global_p95']) if not pd.isna(r['global_p95']) else np.nan

    sub = df_window[(df_window['cmdb_id'] == cmdb) & (df_window['kpi_name'] == kpi)].copy()
    if sub.empty:
        anomaly_count = 0
        earliest_ts = np.nan
        max_val = np.nan
    else:
        # anomaly if value >= global_p95 (use >=); if global_p95 is NaN treat as no anomalies
        if pd.isna(global_p95_val):
            mask = pd.Series([False]*len(sub), index=sub.index)
        else:
            mask = sub['value'] >= global_p95_val
        anomaly_count = int(mask.sum())
        if anomaly_count == 0:
            earliest_ts = np.nan
        else:
            earliest_ts = int(sub.loc[mask, 'timestamp'].min().to_datetime64().astype('int64') // 10**9)
        max_val = float(sub['value'].max())

    summary_rows.append({
        'cmdb_id': cmdb,
        'kpi_name': kpi,
        'global_p95': global_p95_val,
        'anomaly_count': anomaly_count,
        'earliest_anomaly_timestamp': earliest_ts,
        'max_value_in_window': max_val
    })

kpi_summary = pd.DataFrame(summary_rows)[['cmdb_id','kpi_name','global_p95','anomaly_count','earliest_anomaly_timestamp','max_value_in_window']]

# Keep results compact: up to 200 rows for time series, up to 5 rows for summary (we have 5 KPIs)
ts_rows_limited, kpi_summary

```
Out[5]:
```
```python
# Summarize the selected KPI time series and KPI-summary results in plain English.
# Reuse existing variables: ts_rows_limited, kpi_summary

import pandas as pd

# Helper to format unix timestamp (or show 'null')
def fmt_ts(u):
    if pd.isna(u):
        return "null"
    return pd.to_datetime(int(u), unit='s', utc=True).strftime('%Y-%m-%d %H:%M:%S %Z')

n_ts = len(ts_rows_limited) if 'ts_rows_limited' in globals() else 0
kpi_rows = kpi_summary if 'kpi_summary' in globals() else pd.DataFrame()

lines = []
lines.append(f"Time series rows returned for the 5 KPIs in the incident window: {n_ts} rows (limited to 200).")
lines.append("")
lines.append("KPI summary (anomalies within window using precomputed global p95):")
for _, r in kpi_rows.iterrows():
    lines.append(
        f"- {r['cmdb_id']}.{r['kpi_name']}: global_p95={r['global_p95']:.6g}, "
        f"anomaly_count={int(r['anomaly_count'])}, earliest={fmt_ts(r['earliest_anomaly_timestamp'])}, "
        f"max={r['max_value_in_window']:.6g}"
    )

lines.append("")
lines.append("Key takeaways:")
lines.append("- orders.error and orders.workload: each have 2 anomaly rows with max=12.185714, far above their p95s (≈2.43 and ≈4.13). Earliest anomaly at 2024-01-21 08:04:00 UTC — short, high spikes to investigate.")
lines.append("- orders.cpu: 2 anomalies, max=55.454159 vs p95≈41.60, earliest at 2024-01-21 08:04:00 UTC — coincident spike with orders workload/error.")
lines.append("- user.socket: anomaly_count=23 and max equals the global_p95 (21.0) — many samples in the window are at or above the p95, indicating sustained/high socket values.")
lines.append("- catalogue.socket: anomaly_count=25 and max equals the global_p95 (6.0) — all sampled points in the window meet/exceed p95, indicating persistently high socket values during the window.")
lines.append("")
lines.append("Recommendation: prioritize investigating the orders service for the brief but large spikes around 08:04 UTC (error/workload/cpu), and investigate sustained socket pressure on catalogue and user services (persistent at/above p95).")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                    timestamp cmdb_id  kpi_name      value
0   2024-01-21 08:04:00+00:00  orders       cpu  55.454159
1   2024-01-21 08:05:00+00:00  orders       cpu  49.992421
2   2024-01-21 08:06:00+00:00  orders       cpu   8.039083
3   2024-01-21 08:07:00+00:00  orders       cpu   4.959699
4   2024-01-21 08:08:00+00:00  orders       cpu   6.067150
..                        ...     ...       ...        ...
120 2024-01-21 08:24:00+00:00  orders  workload   2.021133
121 2024-01-21 08:25:00+00:00  orders  workload   1.997700
122 2024-01-21 08:26:00+00:00  orders  workload   2.011183
123 2024-01-21 08:27:00+00:00  orders  workload   2.111200
124 2024-01-21 08:28:00+00:00  orders  workload   1.786053

[125 rows x 4 columns],      cmdb_id  kpi_name  global_p95  anomaly_count  earliest_anomaly_timestamp  max_value_in_window
0     orders     error    2.426600              2                  1705824240            12.185714
1     orders  workload    4.132567              2                  1705824240            12.185714
2     orders       cpu   41.601754              2                  1705824240            55.454159
3       user    socket   21.000000             23                  1705824360            21.000000
4  catalogue    socket    6.000000             25                  1705824240             6.000000)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```